In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

def engineer_features_and_train_iforest(df, contamination=0.01, random_state=42):
    """
    Engineer Fourier harmonic features and train Isolation Forest.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Must have columns 'timestamp_utc' and 'Diff'
    contamination : float
        Expected proportion of outliers (default 0.01 = 1%)
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    model : IsolationForest
        Trained model
    df_features : pandas.DataFrame
        DataFrame with engineered features and anomaly scores
    """
    
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Ensure timestamp is datetime
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])

    df.dropna(inplace=True)
    
    # Convert timestamp to seconds since epoch (for periodic calculations)
    df['timestamp_seconds'] = df['timestamp_utc'].astype(np.int64) / 1e9
    
    # Daily period: 86400 seconds (24 hours)
    daily_period = 86400
    df['sin_daily'] = np.sin(2 * np.pi * df['timestamp_seconds'] / daily_period)
    df['cos_daily'] = np.cos(2 * np.pi * df['timestamp_seconds'] / daily_period)
    
    # Weekly period: 604800 seconds (7 days)
    weekly_period = 604800
    df['sin_weekly'] = np.sin(2 * np.pi * df['timestamp_seconds'] / weekly_period)
    df['cos_weekly'] = np.cos(2 * np.pi * df['timestamp_seconds'] / weekly_period)
    
    # Prepare feature matrix
    feature_columns = ['Diff', 'sin_daily', 'cos_daily', 'sin_weekly', 'cos_weekly']
    X = df[feature_columns].values
    
    # Handle potential NaN values in Diff
    if np.isnan(X).any():
        print(f"Warning: Found {np.isnan(X).sum()} NaN values. Filling with 0.")
        X = np.nan_to_num(X, nan=0.0)
    
    # Train Isolation Forest
    model = IsolationForest(
        contamination=contamination,
        random_state=random_state,
        n_estimators=100
    )
    
    # Fit and predict
    df['anomaly_label'] = model.fit_predict(X)  # -1 for anomalies, 1 for normal
    df['anomaly_score'] = model.score_samples(X)  # Lower score = more anomalous
    
    # Create output dataframe with relevant columns
    df_features = df[['timestamp_utc', 'Diff', 'sin_daily', 'cos_daily', 
                       'sin_weekly', 'cos_weekly', 'anomaly_label', 'anomaly_score']]
    
    # Add binary anomaly flag (True/False for easier filtering)
    df_features['is_anomaly'] = df_features['anomaly_label'] == -1
    
    # Summary statistics
    n_anomalies = (df_features['is_anomaly']).sum()
    print(f"\n=== Isolation Forest Training Complete ===")
    print(f"Total samples: {len(df_features)}")
    print(f"Detected anomalies: {n_anomalies} ({100*n_anomalies/len(df_features):.2f}%)")
    print(f"Contamination parameter: {contamination}")
    
    return model, df_features


# Example usage:
#df = pd.DataFrame({
#    'timestamp_utc': pd.date_range('2024-01-01', periods=1000, freq='30min'),
#    'Diff': np.random.randn(1000) * 10 + 50
#})
#
#model, df_with_anomalies = engineer_features_and_train_iforest(df, contamination=0.05)
#
## Get anomalies
#anomalies = df_with_anomalies[df_with_anomalies['is_anomaly']]
#print(anomalies)


In [6]:
df = pd.read_csv('../data_w_diff_001/121771.csv')
model, df_with_anomalies = engineer_features_and_train_iforest(df, contamination=0.01)


=== Isolation Forest Training Complete ===
Total samples: 42578
Detected anomalies: 426 (1.00%)
Contamination parameter: 0.01


C:\Users\Ondřej Černý\AppData\Local\Temp\ipykernel_12408\481756850.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_features['is_anomaly'] = df_features['anomaly_label'] == -1
